## Sửa lỗi Colab không nhận Tesla T4 — 17/11/2026

Phán quyết lỗi `COLAB_T4_REQUIRED`: preflight GPU thất bại **trước optimizer step 1**, vì GPU không phải T4 hoặc CUDA không khả dụng. Phiên cũ có thể đã tạo thư mục `SESSION_LOCK.json` dù chưa chạy bất cứ bước nào. **Không xoá/bỏ qua checkpoint bằng cách sửa tay; dùng một `RUN_ID` mới** khi thử lại.

**Quy trình một lượt an toàn:**
1. Trong Colab, chọn Runtime → Change runtime type → **T4 GPU**, reconnect; kiểm tra `torch.cuda.is_available()` và `torch.cuda.get_device_name(0)`.
2. Chạy notebook mới với `MODE="GPU_PREFLIGHT"` — đọc-only, không tạo session, không mở TinyStories hay checkpoint. Kết quả bắt buộc `LIVE_T4_PREFLIGHT_PASS`.
3. Sau khi PASS, đặt `MODE="ONE_SESSION_40_1024"` và `RUN_ID` mới, chạy lại các code cell. Runner bây giờ kiểm tra T4 **trước khi tạo thư mục phiên**, nên lỗi GPU không làm tiêu thụ RUN_ID lần nữa. Nếu Q4 chưa có, chỉ chạy TECH40 synthetic rồi HOLD trước huấn luyện thật.

Notebook sử dụng **thư mục source cache V2 mới** để không ghi đè runner SHA đã khóa trong cache cũ cùng phiên; vẫn kiểm toàn bộ byte source trước sử dụng. Giữ Q4 HOLD hiện hành; không tự động cấp GO hay mở TEST.


## Đã xác minh Google Drive (17/11/2026)

Tôi đã kiểm tra trực tiếp folder Drive và tải bản sao **read-only** vào môi trường kiểm toán để tái tính SHA-256. Cây thư mục có thật: **Drive của tôi / TTM / ttm-small-pilot/**.

- TRAIN/VAL NPZ: T4_PRIVATE_TRAIN_VAL_WINDOWS.npz, 1.186.107 bytes, SHA256 `184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913` — **PASS_BYTE_SHA**.
- Manifest: T4_PRIVATE_PACK_MANIFEST.json, 1.382 bytes, SHA256 `ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650` — **PASS_BYTE_SHA**.
- Checkpoint gốc: step_1024/checkpoint.pt, 45.074.891 bytes, SHA256 `cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b` — **PASS_BYTE_SHA**.
- READY: READY.json xác nhận checkpoint bước 1024, SHA pack và SHA checkpoint.
- progress.json + science lock: SHA-256 đều khớp READY.

**Trên Colab, bạn chỉ cần mount Drive trước khi chọn PREFLIGHT, TRAIN hoặc ONE_SESSION_40_1024:**

```python
from google.colab import drive
drive.mount("/content/drive")
```

Ba biến `PACK_PATH`, `PACK_MANIFEST_PATH`, `HISTORIC_LEFT_CHECKPOINT` ở code cell đã điền sẵn đúng cây thư mục này, không cần upload ba file lên Colab. `MODE="SOURCE_QA"` mặc định không yêu cầu mount Drive. Notebook không tự truy cập Drive/TEST. Nếu thay vị trí trong Drive, cả ba phải được cập nhật bằng đường dẫn đã kiểm tra SHA, không đoán tên.

**Bảo mật:** Repository GitHub này là public; không công bố ID/URL file Google Drive riêng tư trong notebook. Link Drive cụ thể chỉ trao trong cuộc trò chuyện được kết nối quyền Drive.

**Giới hạn:** Dấu SHA chỉ chứng minh nội dung và provenance checkpoint; không chứng minh quyền thực thi Q4, VRAM hay chất lượng model mới. Mọi dữ liệu riêng tư vẫn nằm trong Google Drive; không commit payload vào GitHub.



# TTM — Một phiên Colab: 40 bước kỹ thuật tổng hợp → 1.024 TinyStories → đối chứng checkpoint

**Thiết kế PM:** một lần thực thi Colab T4 có 2 giai đoạn **độc lập về trọng số**. Trước hết tạo GPT P02 đầy đủ **3.749.376 tham số** và chạy 40 optimizer steps trên 4.096 target token **ngẫu nhiên tổng hợp** mỗi bước, ghi biên nhận kỹ thuật và VRAM/timing. Xóa mô hình synthetic; giai đoạn tiếp theo **khởi tạo GPT mới từ đầu** ở seed lịch sử rồi train 1.024 steps trên exact private TinyStories TRAIN/VAL với BPE2048. Cuối cùng chỉ đánh giá ghép cặp **Small-4M lịch sử @1024 (LEFT)** và **GPT mới @1024 (RIGHT)** trên **256 VAL cũ**, không đụng TEST.

**Điều kiện thực thi:** chế độ `ONE_SESSION_40_1024` cần RUN_ID. Nếu không có exact `OWNER_GO_JSON` đã được quyết định Q4, notebook **chạy 40 bước kỹ thuật rồi STOP trước TinyStories**, tạo `SESSION_STATUS.json` HOLD. Nếu có packet Q4 hợp lệ đã được phân xử riêng, cùng một lệnh sẽ tiếp tục huấn luyện 1.024, tạo READY checkpoint và so sánh hai checkpoint. Notebook không cấp phát GO tự động. Cần owner cung cấp đúng private pack, checkpoint gốc và nơi lưu checkpoint bền vững. `MODE="SOURCE_QA"` vẫn là mặc định không GPU/Torch/corpus.

**Quan trọng:** 40 synthetic steps ở đây dùng **cùng kiến trúc P02 Small-4M**, không phải cấu hình byte-vocab258, context128, 2 layer của bản demo cũ. Không copy optimizer/trọng số synthetic sang GPT TinyStories; hai loss không dùng để so sánh chất lượng khoa học. Bản này vẫn giữ các arm 2.048 trong chế độ TRAIN độc lập cho nghiên cứu tiếp theo; không tự động chạy 2.048 trong phiên 40→1024.

`SOURCE_QA | NATIVE_QA | CAPACITY | PREFLIGHT | TRAIN | RESUME | COMPARE | SYNTHETIC_40 | ONE_SESSION_40_1024` cho phép chọn từng thao tác. `SOURCE_QA` và mã kiểm thử không được trình bày như bằng chứng chạy Colab T4 thực tế.


In [ ]:
import hashlib, json, pathlib, platform, subprocess, sys, urllib.request

MODE = "SOURCE_QA"  # SOURCE_QA | GPU_PREFLIGHT | NATIVE_QA | CAPACITY | PREFLIGHT | TRAIN | RESUME | COMPARE | SYNTHETIC_40 | ONE_SESSION_40_1024
ARM = "replica_1024"  # replica_1024 | control_2048 | replica_2048
RUN_ID = "UNSET_NO_Q4_GO"
RESUME_FROM_STEP = None  # only for RESUME: 256,512,768,1024,1536
WORKSPACE = pathlib.Path("/content/TTM_MATCHED_4M_ONEPACK_CUDA_FIX_V2")
PACK_PATH = "/content/drive/MyDrive/TTM/ttm-small-pilot/t4_private_pack_20261108_v1/T4_PRIVATE_TRAIN_VAL_WINDOWS.npz"
PACK_MANIFEST_PATH = "/content/drive/MyDrive/TTM/ttm-small-pilot/t4_private_pack_20261108_v1/T4_PRIVATE_PACK_MANIFEST.json"
CAPACITY_RECEIPT = "/content/TTM_MATCHED_4M_CAPACITY.json"
OWNER_GO_JSON = "/content/TTM_MATCHED_4M_OWNER_Q4_GO.json"
RUN_ROOT = "/content/TTM_MATCHED_4M_RUNS"
LEFT_CHECKPOINT = "/content/OWNER_CHECKPOINTS/LEFT/checkpoint.pt"
RIGHT_CHECKPOINT = "/content/OWNER_CHECKPOINTS/RIGHT/checkpoint.pt"
COMPARE_RESULT = "/content/TTM_MATCHED_4M_COMPARISON.json"
HISTORIC_LEFT_CHECKPOINT = "/content/drive/MyDrive/TTM/ttm-small-pilot/t4_science_4m_1024_a2_20261108_v1/checkpoints/step_1024/checkpoint.pt"
SESSION_PARENT = "/content/TTM_MATCHED_4M_SESSIONS"

assert platform.python_version_tuple()[:2] == ("3","12"), "PYTHON312_REQUIRED"
assert MODE in {"SOURCE_QA","GPU_PREFLIGHT","NATIVE_QA","CAPACITY","PREFLIGHT","TRAIN","RESUME","COMPARE","SYNTHETIC_40","ONE_SESSION_40_1024"}
RUNNER_COMMIT = "50a5181613840cdbdea71ba78f11987730372f3e"
RUNNER_GIT_BLOB = "933f42b514de5fac650bfa2a421844c51b9b7c64"
SOURCE_COMMIT = "59ee3f30b996405fddc087ab60102b59a021f8ea"
PINNED_SOURCES = {
 "src/p02/transformer_core.py":"9762343e06cf800b50574c2871ae1ed58537fb18",
 "tools/pilot_t4/integrated/t4_capacity_smoke.py":"c9a6f256ba975228683ee7de2a1090e649110e97",
 "tools/pilot_t4/integrated/t4_eval.py":"bef006bdff173fba6d8ba4aa0ef687a1732fd7c4",
 "tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py":"76c4d0155cd0736919f41682f240b47ad3d7c428"
}
print("MODE", MODE, "ARM", ARM, "PYTHON", platform.python_version())


In [ ]:
def exact_git_blob(data: bytes):
    return hashlib.sha1(b"blob " + str(len(data)).encode() + b"\x00" + data).hexdigest()

def fetch_pinned(commit, path, expected, destination):
    url = "https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/" + commit + "/" + path
    with urllib.request.urlopen(url, timeout=35) as response:
        raw = response.read(400_000)
    assert exact_git_blob(raw) == expected, "EXACT_SOURCE_BLOB_MISMATCH:"+path
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == raw, "LOCAL_SOURCE_ALREADY_DIFFERS:"+str(destination)
    else:
        with destination.open("xb") as f:
            f.write(raw)
    return hashlib.sha256(raw).hexdigest()

WORKSPACE.mkdir(parents=True, exist_ok=True)
RUNNER_PATH = WORKSPACE / "src/ttm_new_gpt_baseline/runner.py"
verified = {"runner": fetch_pinned(RUNNER_COMMIT, "src/ttm_new_gpt_baseline/runner.py",
                                   RUNNER_GIT_BLOB, RUNNER_PATH)}
for relative, sha1 in PINNED_SOURCES.items():
    verified[relative] = fetch_pinned(SOURCE_COMMIT, relative, sha1, WORKSPACE/relative)
print("EXACT_SOURCE_PINS_PASS", len(verified))
print(json.dumps(verified, indent=2, sort_keys=True))


In [ ]:
command = [sys.executable, str(RUNNER_PATH)]
if MODE == "SOURCE_QA":
    command += ["qa"]
    timeout_seconds = 60
elif MODE == "GPU_PREFLIGHT":
    # Read-only GPU inspection; no run ID, no session folder or training data.
    command += ["gpu-preflight", "--sources", str(WORKSPACE)]
    timeout_seconds = 60
elif MODE == "NATIVE_QA":
    command += ["native-qa", "--sources", str(WORKSPACE),
                "--out", str(WORKSPACE/"TTM_NATIVE_QA_RECEIPT.json")]
    timeout_seconds = 180
elif MODE == "CAPACITY":
    command += ["capacity", "--sources", str(WORKSPACE), "--out", CAPACITY_RECEIPT]
    timeout_seconds = 210
elif MODE == "SYNTHETIC_40":
    command += ["synthetic-40", "--sources", str(WORKSPACE),
                "--out", str(WORKSPACE/"TTM_SEPARATE_TECHNICAL_40.json")]
    timeout_seconds = 1200
elif MODE == "ONE_SESSION_40_1024":
    assert ARM == "replica_1024", "SESSION_40_1024_ONLY_MATCHED_ARM"
    assert RUN_ID != "UNSET_NO_Q4_GO", "UNIQUE_SESSION_RUN_ID_REQUIRED"
    parent = pathlib.Path(SESSION_PARENT)
    assert not parent.is_symlink(), "SESSION_PARENT_SYMLINK_FORBIDDEN"
    parent.mkdir(parents=True, exist_ok=True)
    session_dir = str(pathlib.Path(SESSION_PARENT) / ("TTM_40_1024__"+RUN_ID))
    command += ["one-session", "--sources", str(WORKSPACE),
                "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH,
                "--arm", ARM, "--run-id", RUN_ID, "--out", session_dir,
                "--historic", HISTORIC_LEFT_CHECKPOINT]
    # Exactly one prospective Q4 covers BOTH 1024 TRAIN and checkpoint COMPARISON.
    # No GO -> synthetic 40 only, with explicit HOLD receipt.
    if pathlib.Path(OWNER_GO_JSON).is_file():
        command += ["--go", OWNER_GO_JSON]
    if RESUME_FROM_STEP is not None:
        command += ["--resume-step", str(RESUME_FROM_STEP)]
    timeout_seconds = 6600
elif MODE == "PREFLIGHT":
    command += ["preflight", "--sources", str(WORKSPACE),
                "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH]
    timeout_seconds = 120
else:
    assert MODE in {"TRAIN","RESUME","COMPARE"}
    assert RUN_ID != "UNSET_NO_Q4_GO", "OWNER_RUN_ID_AND_EXACT_Q4_REQUIRED"
    assert pathlib.Path(OWNER_GO_JSON).is_file(), "EXPLICIT_Q4_GO_FILE_MISSING"
    assert ARM in {"replica_1024","control_2048","replica_2048"}
    if MODE in ("TRAIN","RESUME"):
        command += ["train", "--sources", str(WORKSPACE),
            "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH,
            "--arm", ARM, "--run-id", RUN_ID, "--go", OWNER_GO_JSON,
            "--capacity", CAPACITY_RECEIPT, "--out", RUN_ROOT+"/"+ARM+"__"+RUN_ID]
        if MODE == "RESUME":
            assert RESUME_FROM_STEP in (256,512,768,1024,1536), "EXPLICIT_RESUME_CHECKPOINT_STEP_REQUIRED"
            command += ["--resume-step", str(RESUME_FROM_STEP)]
        timeout_seconds = 5900
    else:
        command += ["compare", "--sources", str(WORKSPACE),
            "--data", PACK_PATH, "--manifest", PACK_MANIFEST_PATH,
            "--arm", ARM, "--run-id", RUN_ID, "--go", OWNER_GO_JSON,
            "--left", LEFT_CHECKPOINT, "--right", RIGHT_CHECKPOINT,
            "--out", COMPARE_RESULT]
        timeout_seconds = 1200
will_attempt_science = MODE in ("TRAIN","RESUME","COMPARE") or (
    MODE == "ONE_SESSION_40_1024" and pathlib.Path(OWNER_GO_JSON).is_file()
)
print("ACTION", MODE, "SCIENCE_REQUESTED", will_attempt_science)
print("GPU_PREFLIGHT_ONLY" if MODE == "GPU_PREFLIGHT" else "NO_AUTOMATIC_GO")
# NO shell=True, no git writes, no automatic Drive mounting or historical TEST read.
result = subprocess.run(command, capture_output=True, text=True,
                        timeout=timeout_seconds, check=False)
print("EXIT_CODE", result.returncode)
print("STDOUT", result.stdout[-18000:])
print("STDERR", result.stderr[-18000:])
assert result.returncode == 0, "FAIL_CLOSED_ACTION_NOT_SUCCESSFUL"


## Đọc biên nhận sau một phiên

- `MODE="SOURCE_QA"` mặc định chỉ chạy kiểm thử nguồn. Muốn chạy kỹ thuật và khoa học liền nhau, chủ động chọn `MODE="ONE_SESSION_40_1024"`, điền RUN_ID mới và chuẩn bị thư mục SESSION_PARENT. Trong toàn bộ thao tác không truy cập TEST.
- **Giai đoạn kỹ thuật:** `TECHNICAL_SYNTHETIC_40.json` xác nhận 40 optimizer steps trên GPT P02 3.749.376 tham số, input ngẫu nhiên, 4.096 labels/step; đo peak VRAM và latency. Sau bước này mô hình kỹ thuật không được checkpoint hay truyền sang TRAIN.
- **Nếu chưa có Q4:** `SESSION_STATUS.json` = `TECHNICAL_40_PASS_SCIENCE_HOLD_NO_Q4`. Không có train hay compare. Đây là hành vi mặc định khi chỉ có thông tin kỹ thuật.
- **Nếu có một Q4 tích hợp hợp lệ:** tiếp tục tạo **mô hình mới từ seed**, huấn luyện 1.024 steps trên private pack TinyStories đã khóa, lưu các READY step 256/512/768/1024 và thực hiện so sánh ghép cặp trên 256 VAL. `SESSION_STATUS.json` cuối vẫn ghi `UNADJUDICATED` cho chất lượng; không kết luận cải thiện tự động.
- **Không được thay đổi cấu hình để tạo thắng lợi giả:** model, số tham số, tokenizer, phân chia câu chuyện, optimizer, 4.096 nhãn/bước và seed lịch sử đều phải trùng với Small-4M. Trọng số synthetic phải được vứt bỏ; mô hình TinyStories phải khởi tạo tươi.
- **Ổn định dữ liệu:** `/content` bị xóa khi reset phiên Colab; đường dẫn SESSION_PARENT nên trỏ đến volume bền vững đã được chuẩn bị và có đủ dung lượng, vì runner không tự mount Drive. Resume phải đúng READY/hash/runID. Không có resume tự động.

**Current status:** chỉ có mã đã triển khai và kiểm thử CI nguồn; **GO khoa học và huấn luyện T4 thực vẫn HOLD/NOT_EXECUTED**.
